> **Aktualna empiryczna Figure 1:** ten historyczny walkthrough korzysta ze starszego samplera. Wykres dla wspólnej siatki k i jawnego eksportu znajdziesz w [`icl_reachability_results.ipynb`](icl_reachability_results.ipynb). Mock i oracle nie są wynikami Luny ani Terry.

# Figure 1: pełny eksperyment osiągalności CLLP

Ten notebook prowadzi przez eksperyment dla losowych podścieżek rozwiązanych trajektorii Sokobana. Z danego kontekstu $s_i$ wybieramy kilka różnych przyszłych stanów $s_{i+k}$, wykonujemy wielokrotnie tę samą goal-conditioned policy i mierzymy, czy dotarła do celu w co najwyżej $L$ akcjach. Później wyliczamy rozkład odległości względnej osobno dla każdego kontekstu i modelu.

**Domyślnie cały walkthrough wykonuje mały eksperyment offline (`mock`).** To deterministyczna heurystyka, a nie odpowiedź Luny ani żadnego innego modelu. Wyniki mock służą do sprawdzenia procesu. Część płatna wymaga jawnego ustawienia obu przełączników w sekcji konfiguracji, osobnego katalogu, klucza API oraz dostępnego budżetu.

Figure 1 w proposal jest ilustracją i zakłada $d=k$. Tutaj $k$ to offset w trajektorii, natomiast $d$ jest *zmierzonym* średnim czasem pierwszego trafienia w udanych próbach; te wartości nie muszą być równe. Notebook nie uruchamia generatora subgoali ani pełnego hierarchical search.

## 1. Środowisko i wybór eksperymentu

Uruchom notebook z katalogu repozytorium `CaRL` albo `CaRL/notebooks`, używając kernela jego `.venv`. Przykładowe dane muszą znajdować się w `data/icl`. `example.json` jest małym pilotem walidacyjnym; `multiscale_validation.json` dodaje trzy rodziny z ilustracji i wymaga znacznie więcej pracy. W tej drugiej konfiguracji większe plansze są syntetyczne, a małe realne.

Aby porównać modele, wpisz ich **sprawdzone identyfikatory API** w `MODEL_IDS`. Ta sama zamrożona lista kontekstów, celów i seedów trafi do wszystkich modeli. Zmiana konfiguracji lub kodu wymaga nowego katalogu wyników. Zanim uruchomisz płatną inferencję, przeczytaj sekcję 2 i upewnij się, że modele są dostępne oraz prawidłowo wycenione w istniejącym ledgerze.

`MODEL_OVERRIDES` pozwala jawnie ustawić dla poszczególnych identyfikatorów `policy_reasoning_effort`, `policy_response_format` i `policy_tokens`; brak wpisu oznacza wartości wspólnego configu. Różne ustawienia mogą mieć wpływ na wynik i koszt — manifest zachowuje je oddzielnie. Przykład z fikcyjnymi identyfikatorami (zastąp je zweryfikowanymi):

```python
MODEL_IDS = ["model-A", "model-B"]
MODEL_OVERRIDES = {"model-B": {"policy_reasoning_effort": "medium", "policy_response_format": "text", "policy_tokens": 4096}}
```

In [ ]:
from pathlib import Path
import json
import os
import hashlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, FileLink

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "carl").is_dir() and (p / "configs/icl").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Uruchom notebook z repozytorium CaRL lub jego podkatalogu.")
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from carl.icl.boards import from_rows, state_hash, to_rows
from carl.icl.datasets import environment, load_data
from carl.icl.config import load_config
from carl.icl.components import ICLComponents
from carl.icl.experiment_api import ExperimentAPI
from scripts.figure1_reachability import Study, MockByteEncoding, experiment_config, freeze, prepare, run, saved_rows
from scripts.figure1_report import write_report

# --- Ustawienia całego walkthrough ---
CONFIG_FILE = ROOT / "configs/icl/figure1/example.json"
PROVIDER = "mock"                 # "mock" lub "openai"
RUN_MOCK = True                   # wykonaj i wznawiaj lokalny eksperyment mock
RUN_PAID = False                  # wyłącznie jawne True uruchamia płatne żądania
MODEL_IDS = None                  # None = modele z configu; np. ["sprawdzone-id-1", "sprawdzone-id-2"]
MODEL_OVERRIDES = None            # opcjonalne ustawienia policy per model; patrz instrukcja
CAMPAIGN_STAGE = "policy_validation"  # musi mieć dostępny budżet w istniejącym ledgerze

# Oddzielne katalogi chronią dane mock przed pomieszaniem z wynikami live.
OUTPUT_DIR = ROOT / "analysis/luna_icl" / f"figure1_walkthrough_v2_{PROVIDER}"
CONTEXT_INDEX = 0                 # kontekst do szczegółowego podglądu
TARGET_INDEX = 0                  # cel w wybranym kontekście
TRIAL_INDEX = 0                   # numer próby do replay
MODEL_INDEX = 0                   # model do replay

if PROVIDER not in {"mock", "openai"}:
    raise ValueError("PROVIDER musi mieć wartość 'mock' albo 'openai'.")
raw = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
raw["output_dir"] = str(OUTPUT_DIR)
if MODEL_IDS is not None:
    raw["models"] = MODEL_IDS
if MODEL_OVERRIDES is not None:
    raw["model_overrides"] = MODEL_OVERRIDES
study = Study.model_validate(raw)
print("Repozytorium:", ROOT)
print("Tryb:", PROVIDER, "—", "DANE MOCK" if PROVIDER == "mock" else "PŁATNE API")
print("Katalog wyników:", OUTPUT_DIR)
display(study.model_dump(mode="json"))

## 2. Plan, liczba prób i kontrola kosztu

Jedna próba może użyć do $L$ zapytań policy (plus ewentualne retry JSON). Wcześniejsze trafienie lub terminal skracają rollout; cache może zmniejszyć liczbę fizycznych żądań. Poniższy wynik to **górna granica liczby logicznych zapytań**, nie wycena.

Dla `openai` istniejący wspólny ledger egzekwuje limit kampanii oraz wybranego etapu. Obejmuje też wcześniejsze koszty i nierozliczone rezerwacje. Nie edytuj jego konfiguracji ani nie zakładaj nowego ledgeru bez uzgodnienia istniejących rozliczeń. Kod odmówi wywołania modelu, którego cena nie jest skonfigurowana. Zgodność ustawień API oraz dostępność modelu muszą być potwierdzone przed płatnym uruchomieniem.

In [ ]:
base = load_config(study.experiment_config)
n_contexts = sum(d.trajectories * d.contexts_per_trajectory for d in study.datasets)
n_targets = n_contexts * study.targets_per_context
n_trials = n_targets * study.trials * len(study.models)
max_calls = n_trials * study.action_cap * (base.json_max_retries + 1)
plan = {
    "contexts": n_contexts, "targets_per_model": n_targets,
    "trials_per_target": study.trials, "planned_trials_all_models": n_trials,
    "window": study.window, "action_cap_L": study.action_cap,
    "max_logical_policy_calls": max_calls,
}
display(pd.Series(plan, name="plan").to_frame())
policy_settings = []
for model in study.models:
    config = experiment_config(study, study.datasets[0], model)
    policy_settings.append({"model": model,
                            "reasoning": config.policy_reasoning_effort or config.reasoning_effort,
                            "format": config.policy_response_format or config.response_format,
                            "max_output_tokens": config.policy_tokens})
print("Rzeczywiste ustawienia żądań policy dla każdego modelu:")
display(pd.DataFrame(policy_settings))
price_config = json.loads((ROOT / "configs/icl/campaign_budget.json").read_text())
configured_models = list(price_config["models"])
print("Modele z ceną w konfiguracji ledgeru:", configured_models)
print("Limit kampanii (USD):", price_config["limit_usd"])
print("Przydział etapu (USD):", price_config["stage_limits_usd"].get(CAMPAIGN_STAGE, "BRAK"))
ledger_path = Path(price_config["ledger_path"])
if not ledger_path.is_absolute():
    ledger_path = ROOT / ledger_path
if ledger_path.exists():
    import sqlite3
    with sqlite3.connect(ledger_path.as_uri() + "?mode=ro", uri=True) as db:
        totals = db.execute("SELECT stage, status, COUNT(*), SUM(COALESCE(charged_nanos, reserved_nanos)) "
                            "FROM attempts GROUP BY stage, status").fetchall()
        effective = db.execute("SELECT value FROM metadata WHERE key='effective_limit_nanos'").fetchone()
    ledger_rows = pd.DataFrame(totals, columns=["stage", "status", "attempts", "committed_nanos"])
    if not ledger_rows.empty:
        ledger_rows["committed_usd"] = ledger_rows.pop("committed_nanos") / 1_000_000_000
        display(ledger_rows)
        print("Łącznie opłaty i rezerwacje (USD):", round(ledger_rows["committed_usd"].sum(), 6))
    if effective:
        print("Efektywny zapisany limit (USD):", int(effective[0]) / 1_000_000_000)
else:
    print("Brak istniejącego ledgeru:", ledger_path)
print("Rezerwacje o nieznanym statusie nie są potwierdzonymi opłatami; wartości nie są prognozą kosztu.")
if PROVIDER == "openai":
    unpriced = [m for m in study.models if m not in configured_models and "*" not in configured_models]
    if unpriced:
        print("BRAK CENY:", unpriced, "— ledger odrzuci płatne żądanie.")
    if not RUN_PAID:
        print("RUN_PAID=False: notebook nie wykona wywołań API.")

## 3. Freeze: trajektorie, konteksty i cele

Przygotowanie jest lokalne: wybiera trajektorie i pozycje losowo bez zwracania, odtwarza świadków w symulatorze oraz zapisuje wybrane stany i hashe danych, kodu, konfiguracji i pakietów. Cele są losowane spośród różnych stanów przyszłych w oknie, z wyłączeniem stanu źródłowego. Jeśli stan powtórzy się w trajektorii, reprezentuje go najwcześniejszy offset. Świadek służy do konstrukcji i weryfikacji celu — jego akcje nie są podawane modelowi jako odpowiedź.

Istniejący manifest musi dokładnie odpowiadać bieżącym wejściom. Jeśli nie odpowiada, wybierz inny `OUTPUT_DIR`; nie nadpisuj wcześniejszego eksperymentu. W trybie live, gdy `RUN_PAID=False`, notebook nie przygotowuje ani nie uruchamia eksperymentu.

In [ ]:
can_run = (PROVIDER == "mock" and RUN_MOCK) or (PROVIDER == "openai" and RUN_PAID)
manifest = None
if can_run:
    if PROVIDER == "openai":
        unpriced = [m for m in study.models if m not in configured_models and "*" not in configured_models]
        if unpriced:
            raise RuntimeError(f"Najpierw uzgodnij wycenę i ledger dla: {unpriced}")
        if CAMPAIGN_STAGE not in price_config["stage_limits_usd"]:
            raise ValueError("Nieznany etap budżetu kampanii.")
        os.environ["CARL_CAMPAIGN_STAGE"] = CAMPAIGN_STAGE
    manifest = prepare(study, PROVIDER)
    print("Freeze gotowy:", OUTPUT_DIR / "manifest.json")
elif (OUTPUT_DIR / "manifest.json").exists():
    manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text(encoding="utf-8"))
    if manifest != freeze(study, PROVIDER):
        raise ValueError("Istniejący freeze nie odpowiada konfiguracji lub kodowi; wybierz inny OUTPUT_DIR.")
    print("Podgląd zweryfikowanego manifestu; wykonanie wyłączone.")
else:
    print("Brak manifestu: włącz RUN_MOCK albo jawnie przygotuj live po kontroli kosztu.")

if manifest is not None:
    print("Provider:", manifest["provider"])
    print("Losowanie:", manifest["sampling"])
    print("Warunek losowości:", manifest["randomness"])
    rows = [{"dataset": c["dataset"], "trajectory_id": c["trajectory_id"],
             "context_id": c["context_id"], "i": c["position"],
             "k": [t["k"] for t in c["targets"]], "source_hash": c["state_hash"][:12]}
            for c in manifest["contexts"]]
    display(pd.DataFrame(rows))

## 4. Obejrzyj $s_i$ oraz $s_{i+k}$

Źródło i cel pochodzą z tej samej zweryfikowanej trajektorii. Wybierz `CONTEXT_INDEX` i `TARGET_INDEX` w pierwszej komórce. $k$ opisuje liczbę akcji w zapisanej trajektorii, a niekoniecznie długość najkrótszej ścieżki.

In [ ]:
TILE_CODES = {"#": 0, "-": 1, ".": 2, "$": 3, "*": 4, "@": 5, "+": 6}
TILE_COLORS = ["#293241", "#f1f3f4", "#f4c95d", "#ac6534", "#d28a38", "#247ba0", "#55a6b9"]
COLOR_SCALE = []
for n, color in enumerate(TILE_COLORS):
    COLOR_SCALE.extend([[n / 7, color], [(n + 1) / 7 - 0.000001, color]])
COLOR_SCALE[-1][0] = 1.0

def board_trace(rows, label):
    chars = [list(row) for row in rows]
    z = [[TILE_CODES[ch] for ch in row] for row in chars]
    return go.Heatmap(z=z, text=chars, zmin=-0.5, zmax=6.5, colorscale=COLOR_SCALE,
                      showscale=False, hovertemplate=f"{label}: %{{text}}<extra></extra>")

def show_pair(source, target, subtitle):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Źródło sᵢ", "Cel sᵢ₊ₖ"))
    fig.add_trace(board_trace(source, "źródło"), row=1, col=1)
    fig.add_trace(board_trace(target, "cel"), row=1, col=2)
    fig.update_yaxes(autorange="reversed", scaleanchor="x", row=1, col=1)
    fig.update_yaxes(autorange="reversed", scaleanchor="x2", row=1, col=2)
    fig.update_layout(title=subtitle, height=440, width=900, margin=dict(t=70))
    fig.show()
    print("Legenda: # ściana · - podłoga · . cel · $ skrzynia · * skrzynia na celu · @ gracz · + gracz na celu")

context = target = None
if manifest is not None:
    context = manifest["contexts"][CONTEXT_INDEX]
    target = context["targets"][TARGET_INDEX]
    show_pair(context["state"], target["state"],
              f"{context['trajectory_id']} · i={context['position']} · k={target['k']}")

## 4a. Co dokładnie dostaje CLLP?

Przed inferencją pokazujemy zbudowane żądanie: instrukcję policy, treningowe przykłady wejście–odpowiedź oraz końcowe zapytanie zawierające $s_i$ i $s_{i+k}$. To tylko konstrukcja payloadu i nie wywołuje API. Model zwraca rozkład czterech akcji, a eksperyment losuje z niego akcję i stosuje ją w symulatorze. Referencyjna sekwencja akcji dla testowej podścieżki nie jest częścią zapytania.

In [ ]:
if context is not None and target is not None:
    dataset = next(d for d in study.datasets if d.name == context["dataset"])
    policy_config = experiment_config(study, dataset, study.models[MODEL_INDEX])
    training, _ = load_data(policy_config)
    # Klient mock służy tylko do konstrukcji wiadomości; nie wykonujemy wywołania.
    inspection_api = ExperimentAPI(policy_config.path("cache_dir"), "mock", policy_config.limits,
                                   encoding=MockByteEncoding())
    policy = ICLComponents(policy_config, inspection_api, environment(from_rows(context["state"])), training)
    payload = policy.policy_payload(from_rows(context["state"]), from_rows(target["state"]))
    print("Model:", payload["model"], "| few-shot:", policy_config.few_shot,
          "| reasoning:", payload["reasoning"], "| fingerprint:", policy.fingerprint())
    print("Rola i długość wiadomości:")
    display(pd.DataFrame([{"role": msg["role"], "characters": len(msg["content"][0]["text"])}
                          for msg in payload["input"]]))
    print("Instrukcja policy:")
    print(payload["input"][0]["content"][0]["text"])
    print("Ostatnie zapytanie (stan i cel):")
    display(json.loads(payload["input"][-1]["content"][0]["text"]))
    if policy_config.few_shot:
        print("Pierwszy przykład treningowy (wejście i odpowiedź):")
        display(json.loads(payload["input"][1]["content"][0]["text"]))
        display(json.loads(payload["input"][2]["content"][0]["text"]))

## 5. Wykonaj lub wznów próby

W trybie `mock` ta komórka uruchamia pełną macierz i wznawia wcześniej ukończony freeze bez powtarzania ukończonych prób. W trybie `openai` kod wykona się wyłącznie przy `RUN_PAID=True`. Każda próba jest zapisywana atomowo; błąd lub brak próby nie oznacza porażki CLLP. Błąd providera zapisany w cache może wymagać uzgodnienia rozliczeń przed wznowieniem.

Powtórzenia losują niezależne akcje z zapisanych rozkładów policy. Cache utrwala odpowiedzi dla identycznych zapytań: nie są to niezależne nowe generacje providera.

In [ ]:
records = []
if manifest is not None:
    if can_run:
        print("Start/wznowienie:", PROVIDER, "— prób:", n_trials)
        run(study, PROVIDER, resume=True)
    records = saved_rows(manifest, OUTPUT_DIR)
    status = pd.DataFrame(records)
    print("Status wszystkich zaplanowanych prób:")
    display(status.groupby(["model", "status"], dropna=False).size().rename("trials").reset_index())
else:
    print("Bez manifestu nie ma prób do uruchomienia ani pokazania.")

## 6. Replay pojedynczego rollouta

Odtwarzamy rzeczywiście zapisane akcje w dokładnym symulatorze, pokazujemy każdy stan i sprawdzamy hash końcowego stanu. Suwak przełącza kolejne kroki; cel jest zawsze pokazany po prawej. `solved_before_target=True` oznacza osiągnięcie zadania w innym stanie przed celem, więc nie jest exact hitem.

In [ ]:
ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]
selected_record = None
if context is not None and target is not None and records:
    selected_model = study.models[MODEL_INDEX]
    matches = [r for r in records if r["model"] == selected_model
               and r["target_id"] == target["target_id"] and r["trial"] == TRIAL_INDEX]
    if not matches or matches[0]["status"] != "complete":
        print("Wybrana próba nie jest ukończona.")
    else:
        selected_record = matches[0]
        board = from_rows(context["state"])
        env = environment(board)
        frames = [to_rows(board)]
        for action in selected_record["actions"]:
            board = np.asarray(env.next_state(board, action))
            frames.append(to_rows(board))
        if state_hash(board) != selected_record["endpoint_hash"]:
            raise ValueError("Replay nie zgadza się z zapisanym stanem końcowym!")
        print("Model:", selected_model, "| k:", target["k"], "| seed:", selected_record["seed"])
        print("Akcje:", [ACTIONS[a] for a in selected_record["actions"]])
        print("Exact hit:", selected_record["exact_hit"], "| first hit:", selected_record["first_hit"],
              "| terminal przed celem:", selected_record["solved_before_target"])
        print("Replay hash końca: poprawny")
        fig = make_subplots(rows=1, cols=2, subplot_titles=("Stan rollouta", "Docelowy sᵢ₊ₖ"))
        fig.add_trace(board_trace(frames[0], "rollout"), row=1, col=1)
        fig.add_trace(board_trace(target["state"], "cel"), row=1, col=2)
        fig.update_yaxes(autorange="reversed", scaleanchor="x", row=1, col=1)
        fig.update_yaxes(autorange="reversed", scaleanchor="x2", row=1, col=2)
        fig.frames = [go.Frame(name=str(i), data=[board_trace(rows, "rollout")], traces=[0])
                      for i, rows in enumerate(frames)]
        steps = [dict(label=str(i), method="animate", args=[[str(i)],
                 {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}])
                 for i in range(len(frames))]
        fig.update_layout(title="Replay: przesuń suwak, aby obejrzeć kolejne stany", height=500, width=900,
                          sliders=[dict(active=0, currentvalue={"prefix": "Krok: "}, steps=steps)])
        fig.show()

## 7. Raport: $\hat p$, $d$ i $u$

Raport zapisuje wszystkie zaplanowane próby, w tym brakujące, oraz podsumowanie każdego celu. $\hat p=\mathrm{hits}/R$ jest publikowane dopiero po ukończeniu wszystkich $R$ prób celu. $d$ to średni first hit tylko w sukcesach. Przy zerze sukcesów $d$ i $u$ są puste. Jeśli choć jeden cel w kontekście nie jest kompletny, nie liczymy $u$ dla całego kontekstu, bo rozkład referencyjny byłby niepełny.

In [ ]:
targets = pd.DataFrame()
if manifest is not None:
    target_rows = write_report(records, OUTPUT_DIR, study.trials, provider=manifest["provider"])
    targets = pd.DataFrame(target_rows)
    if not targets.empty:
        display(targets.groupby(["model", "status"], dropna=False).size().rename("targets").reset_index())
        columns = [c for c in ("model", "dataset", "context_id", "position", "k", "status",
                   "completed_trials", "expected_trials", "hits", "p_hat", "d", "weight", "u") if c in targets]
        display(targets[columns].sort_values(["model", "context_id", "k"]))
    print("CSV i figury zapisane w", OUTPUT_DIR)
else:
    print("Brak danych do raportu.")

## 8. Wykresy jak w Figure 1, bez założenia $d=k$

Po lewej: profil prawdopodobieństwa dokładnego trafienia względem offsetu trajektorii $k$. Po prawej: percentyl $u$ względem **zmierzonego** $d$. Każda seria oznacza pojedynczy kontekst i model; mały pilot nie jest estymatą całej populacji. Nie zakładamy monotonicznej osiągalności.

In [ ]:
if not targets.empty:
    complete = targets.loc[targets["status"].eq("complete")].copy()
    if complete.empty:
        print("Brak kompletnych celów do wykreślenia.")
    else:
        complete["series"] = complete["model"].astype(str) + " / " + complete["context_id"].astype(str)
        fig = px.scatter(complete.sort_values("k"), x="k", y="p_hat", color="series",
                         hover_data=["dataset", "context_id", "k", "hits", "d", "u"],
                         title=f"CLLP: exact-hit rate względem offsetu k [{manifest['provider'].upper()}]")
        fig.update_yaxes(range=[-0.03, 1.03], title="p̂ = exact hits / R")
        fig.update_xaxes(title="Offset k w trajektorii")
        fig.show()
        labeled = complete.dropna(subset=["d", "u"])
        if not labeled.empty:
            fig = px.scatter(labeled, x="d", y="u", color="series", symbol="dataset",
                             hover_data=["context_id", "k", "hits", "p_hat", "weight"],
                             title=f"Ważony percentyl względem zmierzonego first-hit time [{manifest['provider'].upper()}]")
            fig.update_yaxes(range=[-0.03, 1.03], title="Percentyl u")
            fig.update_xaxes(title="Średni first-hit time d (tylko sukcesy)")
            fig.show()
        else:
            print("Brak wystarczających pełnych danych do wyznaczenia percentyli.")

## 9. Skąd bierze się percentyl? Sprawdzenie jednego kontekstu

Wagi celów to $w_j=\hat p_j/\sum_m \hat p_m$. Przy tym samym $R$ są proporcjonalne do liczby trafień. Cele o takim samym $d$ dzielą przedział ważonej dystrybuanty i otrzymują jego środek. Tabela pokazuje również cele bez trafień; nie wchodzą do mianownika rozkładu.

In [ ]:
if context is not None and not targets.empty:
    selected_model = study.models[MODEL_INDEX]
    local = targets.loc[(targets["model"] == selected_model)
                        & (targets["context_id"] == context["context_id"])].copy()
    if not local.empty:
        display(local[["k", "status", "hits", "p_hat", "d", "weight", "u"]].sort_values("k"))
        if local["status"].eq("complete").all() and local["hits"].sum() > 0:
            distribution = (local.loc[local["hits"] > 0].groupby("d", as_index=False)["weight"].sum()
                            .sort_values("d"))
            distribution["cdf_at"] = distribution["weight"].cumsum()
            distribution["cdf_before"] = distribution["cdf_at"] - distribution["weight"]
            distribution["midpoint_u"] = (distribution["cdf_before"] + distribution["cdf_at"]) / 2
            print("Kontrola ważonej dystrybuanty (wiązania d połączone):")
            display(distribution)

## 10. Porównanie modeli na tych samych celach

Jeśli skonfigurowano więcej modeli, tabela łączy wyniki po `target_id` i zachowuje wyłącznie pary z kompletnymi próbami. Różnica dotyczy estimated hit rate na tym samym celu, nie porównania niezależnych losowych plansz. Przy małej liczbie kontekstów nie należy traktować średniej jako wyniku o precyzji publikacyjnej. Różne modele mogą mieć różne konfiguracje zgodności API; ten notebook nie potwierdza dostępności ani cen modeli.

In [ ]:
if targets.empty or len(study.models) < 2:
    print("Porównanie wymaga co najmniej dwóch skonfigurowanych modeli i wyników.")
else:
    observed = targets.loc[targets["status"].eq("complete"), ["target_id", "context_id", "k", "model", "p_hat"]]
    paired = observed.pivot(index=["target_id", "context_id", "k"], columns="model", values="p_hat")
    paired = paired.dropna(subset=study.models)
    print("Kompletne sparowane cele:", len(paired))
    if not paired.empty:
        for model in study.models[1:]:
            paired[f"{model} − {study.models[0]}"] = paired[model] - paired[study.models[0]]
        display(paired.reset_index())
        print("Średnia różnica dla obserwowanych celów:")
        display(paired[[c for c in paired if " − " in str(c)]].mean().to_frame("mean_delta"))

## 11. Pliki, interpretacja i następny eksperyment

Manifest i próby pozostają lokalne w `analysis/`, który jest ignorowany przez Git. Zarchiwizuj je razem z cache i wersją repozytorium, jeśli mają stanowić trwały wynik. Odseparuj dane mock, validation i końcowe dane testowe. Kontroluj liczbę kontekstów, liczbę prób, niepełne wyniki, koszt i wpływ doboru plansz przed wyciągnięciem wniosków.

Pełne pytanie badawcze proposal wymaga również niezależnej oceny stabilności percentyli, porównania celów dostarczonych i wygenerowanych oraz osobnych eksperymentów hierarchical search. Ten walkthrough ich nie zastępuje.

In [ ]:
if manifest is not None:
    for name in ["manifest.json", "results.csv", "targets.csv", "reachability.pdf", "reachability.svg", "percentile.pdf", "percentile.svg"]:
        file = OUTPUT_DIR / name
        if file.exists():
            display(FileLink(str(file)))
    print("Provider wyników:", manifest["provider"],
          "— MOCK nie jest pomiarem modelu" if manifest["provider"] == "mock" else "— dane API")